In [1]:
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

# Target
target = "satisfaction"

# Features
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols).copy()
y = train[target].astype(int)

X_test = test.drop(columns=["id"]).copy()

# Categorical columns
cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Same validation split for every model
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (559708, 21)
Valid: (139927, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


===========================================**LIGHT LGM**======================================

In [4]:
import lightgbm as lgb

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_valid_lgb[col] = X_valid_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

In [5]:
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=SEED,
    n_jobs=-1
)

In [6]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import roc_auc_score

leaf_values = [15, 31, 95, 127, 255]

lgb_leaf_results = []

for leaves in leaf_values:

    print("\n" + "=" * 70)
    print(f"LGB EXP | num_leaves={leaves}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=leaves,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]

    auc = roc_auc_score(y_valid, pred)

    print(f"num_leaves : {leaves}")
    print(f"best_iter  : {model.best_iteration_}")
    print(f"AUC        : {auc:.8f}")
    print(f"Δ baseline : {auc - 0.957875:.8f}")

    lgb_leaf_results.append({
        "num_leaves": leaves,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta": auc - 0.957875
    })

lgb_leaf_results


LGB EXP | num_leaves=15


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 15
best_iter  : 2994
AUC        : 0.95751697
Δ baseline : -0.00035803

LGB EXP | num_leaves=31


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023602 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 31
best_iter  : 1607
AUC        : 0.95765979
Δ baseline : -0.00021521

LGB EXP | num_leaves=95


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028334 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 127
best_iter  : 555
AUC        : 0.95813478
Δ baseline : 0.00025978

LGB EXP | num_leaves=255


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024203 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 255
best_iter  : 286
AUC        : 0.95800192
Δ baseline : 0.00012692


[{'num_leaves': 15,
  'best_iteration': 2994,
  'auc': np.float64(0.9575169682608711),
  'delta': np.float64(-0.00035803173912896114)},
 {'num_leaves': 31,
  'best_iteration': 1607,
  'auc': np.float64(0.957659785749676),
  'delta': np.float64(-0.00021521425032400288)},
 {'num_leaves': 95,
  'best_iteration': 729,
  'auc': np.float64(0.9579896211790947),
  'delta': np.float64(0.00011462117909466496)},
 {'num_leaves': 127,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta': np.float64(0.00025977997721904345)},
 {'num_leaves': 255,
  'best_iteration': 286,
  'auc': np.float64(0.9580019171657562),
  'delta': np.float64(0.00012691716575619072)}]

In [9]:
leaf_values = [105, 115, 127, 140, 155, 175, 195]

lgb_leaf_zoom_results = []

for leaves in leaf_values:

    print("\n" + "=" * 70)
    print(f"LGB LEAF ZOOM | num_leaves={leaves}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=leaves,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"num_leaves : {leaves}")
    print(f"best_iter  : {model.best_iteration_}")
    print(f"AUC        : {auc:.8f}")
    print(f"Δ baseline : {auc - 0.957875:.8f}")

    lgb_leaf_zoom_results.append({
        "num_leaves": leaves,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta": auc - 0.957875
    })

lgb_leaf_zoom_results


LGB LEAF ZOOM | num_leaves=105


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027082 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 105
best_iter  : 559
AUC        : 0.95795834
Δ baseline : 0.00008334

LGB LEAF ZOOM | num_leaves=115


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025796 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 115
best_iter  : 512
AUC        : 0.95800349
Δ baseline : 0.00012849

LGB LEAF ZOOM | num_leaves=127


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 175
best_iter  : 462
AUC        : 0.95802274
Δ baseline : 0.00014774

LGB LEAF ZOOM | num_leaves=195


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024431 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
num_leaves : 195
best_iter  : 587
AUC        : 0.95799028
Δ baseline : 0.00011528


[{'num_leaves': 105,
  'best_iteration': 559,
  'auc': np.float64(0.9579583382281587),
  'delta': np.float64(8.3338228158647e-05)},
 {'num_leaves': 115,
  'best_iteration': 512,
  'auc': np.float64(0.9580034901440403),
  'delta': np.float64(0.00012849014404026082)},
 {'num_leaves': 127,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta': np.float64(0.00025977997721904345)},
 {'num_leaves': 140,
  'best_iteration': 513,
  'auc': np.float64(0.9580817517887688),
  'delta': np.float64(0.00020675178876872202)},
 {'num_leaves': 155,
  'best_iteration': 519,
  'auc': np.float64(0.9580475150157564),
  'delta': np.float64(0.00017251501575632489)},
 {'num_leaves': 175,
  'best_iteration': 462,
  'auc': np.float64(0.9580227392886059),
  'delta': np.float64(0.00014773928860589702)},
 {'num_leaves': 195,
  'best_iteration': 587,
  'auc': np.float64(0.957990280354563),
  'delta': np.float64(0.00011528035456298547)}]

In [10]:
child_values = [20, 25, 50, 75, 100, 150, 200, 300]

lgb_child_results = []

for child in child_values:

    print("\n" + "=" * 70)
    print(f"LGB CHILD | min_child_samples={child}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=child,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"min_child_samples : {child}")
    print(f"best_iter         : {model.best_iteration_}")
    print(f"AUC               : {auc:.8f}")
    print(f"Δ baseline        : {auc - 0.957875:.8f}")
    print(f"Δ current champ   : {auc - 0.9581347799772191:.8f}")

    lgb_child_results.append({
        "min_child_samples": child,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta_baseline": auc - 0.957875,
        "delta_champion": auc - 0.9581347799772191
    })

lgb_child_results


LGB CHILD | min_child_samples=20


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050570 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
min_child_samples : 20
best_iter         : 555
AUC               : 0.95813478
Δ baseline        : 0.00025978
Δ current champ   : 0.00000000

LGB CHILD | min_child_samples=25


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
min_child_samples : 25
best_iter         : 585
AUC               : 0.95802377
Δ baseline        : 0.00014877
Δ current champ   : -0.00011101

LGB CHILD | min_child_samples=50


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053608 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
min_child_samples : 200
best_iter         : 551
AUC               : 0.95811925
Δ baseline        : 0.00024425
Δ current champ   : -0.00001553

LGB CHILD | min_child_samples=300


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025849 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
min_child_samples : 300
best_iter         : 578
AUC               : 0.95800285
Δ baseline        : 0.00012785
Δ current champ   : -0.00013193


[{'min_child_samples': 20,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_baseline': np.float64(0.00025977997721904345),
  'delta_champion': np.float64(0.0)},
 {'min_child_samples': 25,
  'best_iteration': 585,
  'auc': np.float64(0.9580237679375634),
  'delta_baseline': np.float64(0.00014876793756335438),
  'delta_champion': np.float64(-0.00011101203965568907)},
 {'min_child_samples': 50,
  'best_iteration': 460,
  'auc': np.float64(0.9579559795367202),
  'delta_baseline': np.float64(8.097953672014935e-05),
  'delta_champion': np.float64(-0.0001788004404988941)},
 {'min_child_samples': 75,
  'best_iteration': 560,
  'auc': np.float64(0.9579843536728516),
  'delta_baseline': np.float64(0.00010935367285158648),
  'delta_champion': np.float64(-0.00015042630436745696)},
 {'min_child_samples': 100,
  'best_iteration': 556,
  'auc': np.float64(0.9580218738039421),
  'delta_baseline': np.float64(0.0001468738039420714),
  'delta_champion': np.float64(-0.00011290617

In [12]:
learning_rates = [
    (0.01, 5000),
    (0.02, 4000),
    (0.03, 3000),
    (0.05, 2500),
    (0.075, 2000),
]

lgb_lr_results = []

for lr, n_estimators in learning_rates:

    print("\n" + "=" * 70)
    print(f"LGB LR | learning_rate={lr} | n_estimators={n_estimators}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        learning_rate=lr,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"learning_rate : {lr}")
    print(f"best_iter     : {model.best_iteration_}")
    print(f"AUC           : {auc:.8f}")
    print(f"Δ champion    : {auc - 0.9581347799772191:.8f}")

    lgb_lr_results.append({
        "learning_rate": lr,
        "max_estimators": n_estimators,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta_champion": auc - 0.9581347799772191
    })

lgb_lr_results


LGB LR | learning_rate=0.01 | n_estimators=5000


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046671 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
learning_rate : 0.01
best_iter     : 1716
AUC           : 0.95811273
Δ champion    : -0.00002205

LGB LR | learning_rate=0.02 | n_estimators=4000


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024930 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
learning_rate : 0.02
best_iter     : 1016
AUC           : 0.95807010
Δ champion    : -0.00006468

LGB LR | learning_rate=0.03 | n_estimators=3000


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030767 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
learning_rate : 0.05
best_iter     : 348
AUC           : 0.95786479
Δ champion    : -0.00026999

LGB LR | learning_rate=0.075 | n_estimators=2000


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043625 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
learning_rate : 0.075
best_iter     : 145
AUC           : 0.95783343
Δ champion    : -0.00030135


[{'learning_rate': 0.01,
  'max_estimators': 5000,
  'best_iteration': 1716,
  'auc': np.float64(0.958112733553111),
  'delta_champion': np.float64(-2.2046424108079776e-05)},
 {'learning_rate': 0.02,
  'max_estimators': 4000,
  'best_iteration': 1016,
  'auc': np.float64(0.9580701005959416),
  'delta_champion': np.float64(-6.467938127752149e-05)},
 {'learning_rate': 0.03,
  'max_estimators': 3000,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'learning_rate': 0.05,
  'max_estimators': 2500,
  'best_iteration': 348,
  'auc': np.float64(0.9578647894238295),
  'delta_champion': np.float64(-0.00026999055338960964)},
 {'learning_rate': 0.075,
  'max_estimators': 2000,
  'best_iteration': 145,
  'auc': np.float64(0.9578334285637542),
  'delta_champion': np.float64(-0.0003013514134648787)}]

In [13]:
subsample_values = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 1.00]

lgb_subsample_results = []

for subsample in subsample_values:

    print("\n" + "=" * 70)
    print(f"LGB SUBSAMPLE | subsample={subsample}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,
        subsample=subsample,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"subsample  : {subsample}")
    print(f"best_iter  : {model.best_iteration_}")
    print(f"AUC        : {auc:.8f}")
    print(f"Δ champion : {auc - 0.9581347799772191:.8f}")

    lgb_subsample_results.append({
        "subsample": subsample,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta_champion": auc - 0.9581347799772191
    })

lgb_subsample_results


LGB SUBSAMPLE | subsample=0.6


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029827 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
subsample  : 0.6
best_iter  : 555
AUC        : 0.95813478
Δ champion : 0.00000000

LGB SUBSAMPLE | subsample=0.7


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.075649 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
subsample  : 0.7
best_iter  : 555
AUC        : 0.95813478
Δ champion : 0.00000000

LGB SUBSAMPLE | subsample=0.8


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032719 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
subsample  : 0.95
best_iter  : 555
AUC        : 0.95813478
Δ champion : 0.00000000

LGB SUBSAMPLE | subsample=1.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026029 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
subsample  : 1.0
best_iter  : 555
AUC        : 0.95813478
Δ champion : 0.00000000


[{'subsample': 0.6,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 0.7,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 0.8,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 0.85,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 0.9,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 0.95,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'subsample': 1.0,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)}]

In [14]:
colsample_values = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 1.00]

lgb_colsample_results = []

for colsample in colsample_values:

    print("\n" + "=" * 70)
    print(f"LGB COLSAMPLE | colsample_bytree={colsample}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=colsample,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"colsample  : {colsample}")
    print(f"best_iter  : {model.best_iteration_}")
    print(f"AUC        : {auc:.8f}")
    print(f"Δ champion : {auc - 0.9581347799772191:.8f}")

    lgb_colsample_results.append({
        "colsample_bytree": colsample,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta_champion": auc - 0.9581347799772191
    })

lgb_colsample_results


LGB COLSAMPLE | colsample_bytree=0.6


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.047479 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
colsample  : 0.6
best_iter  : 907
AUC        : 0.95801428
Δ champion : -0.00012050

LGB COLSAMPLE | colsample_bytree=0.7


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031004 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
colsample  : 0.7
best_iter  : 605
AUC        : 0.95807489
Δ champion : -0.00005989

LGB COLSAMPLE | colsample_bytree=0.8


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
colsample  : 0.95
best_iter  : 531
AUC        : 0.95785106
Δ champion : -0.00028372

LGB COLSAMPLE | colsample_bytree=1.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026711 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
colsample  : 1.0
best_iter  : 474
AUC        : 0.95783855
Δ champion : -0.00029623


[{'colsample_bytree': 0.6,
  'best_iteration': 907,
  'auc': np.float64(0.9580142810254532),
  'delta_champion': np.float64(-0.00012049895176591274)},
 {'colsample_bytree': 0.7,
  'best_iteration': 605,
  'auc': np.float64(0.9580748866795736),
  'delta_champion': np.float64(-5.989329764544138e-05)},
 {'colsample_bytree': 0.8,
  'best_iteration': 497,
  'auc': np.float64(0.9580170932039547),
  'delta_champion': np.float64(-0.00011768677326440269)},
 {'colsample_bytree': 0.85,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'colsample_bytree': 0.9,
  'best_iteration': 523,
  'auc': np.float64(0.9579360690439211),
  'delta_champion': np.float64(-0.00019871093329792977)},
 {'colsample_bytree': 0.95,
  'best_iteration': 531,
  'auc': np.float64(0.9578510572396044),
  'delta_champion': np.float64(-0.0002837227376146778)},
 {'colsample_bytree': 1.0,
  'best_iteration': 474,
  'auc': np.float64(0.9578385499774276),
  'delta_champion': n

In [15]:
alpha_values = [0.0, 0.01, 0.05, 0.10, 0.20, 0.50, 1.00]

lgb_alpha_results = []

for alpha in alpha_values:

    print("\n" + "=" * 70)
    print(f"LGB ALPHA | reg_alpha={alpha}")
    print("=" * 70)

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=alpha,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, pred)

    print(f"reg_alpha  : {alpha}")
    print(f"best_iter  : {model.best_iteration_}")
    print(f"AUC        : {auc:.8f}")
    print(f"Δ champion : {auc - 0.9581347799772191:.8f}")

    lgb_alpha_results.append({
        "reg_alpha": alpha,
        "best_iteration": model.best_iteration_,
        "auc": auc,
        "delta_champion": auc - 0.9581347799772191
    })

lgb_alpha_results


LGB ALPHA | reg_alpha=0.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063508 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
reg_alpha  : 0.0
best_iter  : 568
AUC        : 0.95800216
Δ champion : -0.00013262

LGB ALPHA | reg_alpha=0.01


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084999 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
reg_alpha  : 0.01
best_iter  : 506
AUC        : 0.95796706
Δ champion : -0.00016772

LGB ALPHA | reg_alpha=0.05


...

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063326 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
reg_alpha  : 0.5
best_iter  : 640
AUC        : 0.95801553
Δ champion : -0.00011925

LGB ALPHA | reg_alpha=1.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034710 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
reg_alpha  : 1.0
best_iter  : 504
AUC        : 0.95807115
Δ champion : -0.00006363


[{'reg_alpha': 0.0,
  'best_iteration': 568,
  'auc': np.float64(0.9580021605154191),
  'delta_champion': np.float64(-0.00013261946179998763)},
 {'reg_alpha': 0.01,
  'best_iteration': 506,
  'auc': np.float64(0.9579670585681379),
  'delta_champion': np.float64(-0.0001677214090811674)},
 {'reg_alpha': 0.05,
  'best_iteration': 529,
  'auc': np.float64(0.958022583366862),
  'delta_champion': np.float64(-0.00011219661035710704)},
 {'reg_alpha': 0.1,
  'best_iteration': 555,
  'auc': np.float64(0.9581347799772191),
  'delta_champion': np.float64(0.0)},
 {'reg_alpha': 0.2,
  'best_iteration': 468,
  'auc': np.float64(0.9579698630902299),
  'delta_champion': np.float64(-0.00016491688698916374)},
 {'reg_alpha': 0.5,
  'best_iteration': 640,
  'auc': np.float64(0.9580155311929599),
  'delta_champion': np.float64(-0.00011924878425917385)},
 {'reg_alpha': 1.0,
  'best_iteration': 504,
  'auc': np.float64(0.9580711516968025),
  'delta_champion': np.float64(-6.362828041661661e-05)}]

In [16]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import roc_auc_score

# ============================================================
# REG_LAMBDA SWEEP
# ============================================================

# Use the best alpha from the previous reg_alpha sweep
BEST_ALPHA = 0.1   # <-- change this after alpha sweep

lambda_values = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]

lambda_results = []

for reg_lambda in lambda_values:

    print(f"\n{'='*70}")
    print(f"Testing reg_lambda = {reg_lambda}")
    print(f"{'='*70}")

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,

        # Current LightGBM champion
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,

        subsample=0.85,
        colsample_bytree=0.85,

        # Regularization
        reg_alpha=BEST_ALPHA,
        reg_lambda=reg_lambda,

        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    valid_pred = model.predict_proba(X_valid_lgb)[:, 1]
    auc = roc_auc_score(y_valid, valid_pred)

    result = {
        "reg_lambda": reg_lambda,
        "best_iteration": model.best_iteration_,
        "auc": auc
    }

    lambda_results.append(result)

    print(
        f"lambda={reg_lambda:<5} | "
        f"best_iter={model.best_iteration_:<5} | "
        f"AUC={auc:.9f}"
    )


# ============================================================
# RESULTS
# ============================================================

lambda_results_sorted = sorted(
    lambda_results,
    key=lambda x: x["auc"],
    reverse=True
)

print("\n" + "="*70)
print("REG_LAMBDA RESULTS")
print("="*70)

for r in lambda_results_sorted:
    print(
        f"lambda={r['reg_lambda']:<5} | "
        f"iter={r['best_iteration']:<5} | "
        f"AUC={r['auc']:.9f}"
    )

best_lambda_result = lambda_results_sorted[0]

print("\n" + "="*70)
print("BEST REG_LAMBDA")
print("="*70)

print(f"Best lambda     : {best_lambda_result['reg_lambda']}")
print(f"Best iteration  : {best_lambda_result['best_iteration']}")
print(f"Best AUC        : {best_lambda_result['auc']:.9f}")


Testing reg_lambda = 0.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031138 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
lambda=0.0   | best_iter=499   | AUC=0.957948928

Testing reg_lambda = 0.05


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030318 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
lambda=0.05  | best_iter=556   | AUC=0.958075497

Testing reg_lambda = 0.1


...

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031672 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
lambda=1.0   | best_iter=539   | AUC=0.958030559

Testing reg_lambda = 2.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024101 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
lambda=2.0   | best_iter=543   | AUC=0.958056176

Testing reg_lambda = 5.0


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056186 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
lambda=5.0   | best_iter=546   | AUC=0.958078789

REG_LAMBDA RESULTS
lambda=0.5   | iter=555   | AUC=0.958134780
lambda=5.0   | iter=546   | AUC=0.958078789
lambda=0.05  | iter=556   | AUC=0.958075497
lambda=2.0   | iter=543   | AUC=0.958056176
lambda=1.0   | iter=539   | AUC=0.958030559
lambda=0.25  | iter=450   | AUC=0.958006497
lambda=0.1 

In [20]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import roc_auc_score

# ============================================================
# MAX_DEPTH SWEEP
# ============================================================

depth_values = [-1, 5, 6, 7, 8, 9, 10, 12]

depth_results = []

for max_depth in depth_values:

    print(f"\n{'='*70}")
    print(f"Testing max_depth = {max_depth}")
    print(f"{'='*70}")

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,

        # Current champion
        num_leaves=127,
        min_child_samples=20,

        # Parameter being tested
        max_depth=max_depth,

        subsample=0.85,
        colsample_bytree=0.85,

        reg_alpha=0.1,
        reg_lambda=0.5,

        random_state=SEED,
        n_jobs=-1,

        verbosity=-1
    )

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=cat_cols,
        eval_set=[(X_valid_lgb, y_valid)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    valid_pred = model.predict_proba(X_valid_lgb)[:, 1]

    auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    result = {
        "max_depth": max_depth,
        "best_iteration": model.best_iteration_,
        "auc": auc
    }

    depth_results.append(result)

    print(
        f"depth={max_depth:<3} | "
        f"best_iter={model.best_iteration_:<5} | "
        f"AUC={auc:.9f}"
    )


# ============================================================
# SORT RESULTS
# ============================================================

depth_results_sorted = sorted(
    depth_results,
    key=lambda x: x["auc"],
    reverse=True
)

print("\n" + "="*70)
print("MAX_DEPTH RESULTS")
print("="*70)

for r in depth_results_sorted:

    delta = r["auc"] - 0.9581347799772191

    print(
        f"depth={r['max_depth']:<3} | "
        f"iter={r['best_iteration']:<5} | "
        f"AUC={r['auc']:.9f} | "
        f"delta={delta:+.9f}"
    )


# ============================================================
# BEST
# ============================================================

best_depth = depth_results_sorted[0]

print("\n" + "="*70)
print("BEST MAX_DEPTH")
print("="*70)

print(f"Best depth      : {best_depth['max_depth']}")
print(f"Best iteration  : {best_depth['best_iteration']}")
print(f"Best AUC        : {best_depth['auc']:.9f}")
print(
    f"Delta champion  : "
    f"{best_depth['auc'] - 0.9581347799772191:+.9f}"
)


Testing max_depth = -1


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


depth=-1  | best_iter=555   | AUC=0.958134780

Testing max_depth = 5


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


depth=5   | best_iter=2999  | AUC=0.957576141

Testing max_depth = 6


...

depth=9   | best_iter=747   | AUC=0.957883144

Testing max_depth = 10


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


depth=10  | best_iter=720   | AUC=0.957864894

Testing max_depth = 12


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


depth=12  | best_iter=854   | AUC=0.958053753

MAX_DEPTH RESULTS
depth=-1  | iter=555   | AUC=0.958134780 | delta=+0.000000000
depth=12  | iter=854   | AUC=0.958053753 | delta=-0.000081027
depth=9   | iter=747   | AUC=0.957883144 | delta=-0.000251636
depth=10  | iter=720   | AUC=0.957864894 | delta=-0.000269886
depth=8   | iter=814   | AUC=0.957780226 | delta=-0.000354554
depth=7   | iter=896   | AUC=0.957681015 | delta=-0.000453765
depth=5   | iter=2999  | AUC=0.957576141 | delta=-0.000558639
depth=6   | iter=1603  | AUC=0.957523071 | delta=-0.000611709

BEST MAX_DEPTH
Best depth      : -1
Best iteration  : 555
Best AUC        : 0.958134780
Delta champion  : +0.000000000


### BEST CASE => LGB
num_leaves = 127

min_child_samples = 20,
        
max_depth = -1,

subsample = 0.85,

colsample_bytree = 0.85,

reg_alpha = 0.1,

reg_lambda = 0.5,

In [26]:
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_valid_lgb, y_valid)],
    callbacks=[
        lgb.early_stopping(100, verbose=False),
        lgb.log_evaluation(0)
    ]
)

lgb_valid_pred = lgb_model.predict_proba(X_valid_lgb)[:, 1]
lgb_test_pred = lgb_model.predict_proba(X_test_lgb)[:, 1]

lgb_auc = roc_auc_score(y_valid, lgb_valid_pred)

print(f"Final LightGBM AUC: {lgb_auc:.9f}")
print(f"Best iteration: {lgb_model.best_iteration_}")

np.save(
    "artifacts/oof/lgb_optimized.npy",
    lgb_valid_pred
)

np.save(
    "artifacts/test/lgb_optimized.npy",
    lgb_test_pred
)

lgb_model.booster_.save_model(
    "artifacts/lgb_optimized.txt"
)

print("Saved final LightGBM artifacts.")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Final LightGBM AUC: 0.958134780
Best iteration: 555
Saved final LightGBM artifacts.
